# Attr-Only Elbow Like-Game Clustering

This notebook clusters games using structured `game_attr` attributes only. It intentionally excludes cabinet-name features, performance-derived features, `game_type`, raw theme-name text/token features, and duplicate flattened game-matrix indicators.

A clean semantic theme category is included as one derived attr feature. It is built only from true theme/name fields (`theme_name`, `theme_name_friendly`, `family_name`, and `brand_name`), so product-format labels such as `Format Non Theme` are not used.

The relationship map is used only to keep accepted attr-to-historical-game links and to produce the final like-game mapping table. Low-confidence review/fuzzy/no-match relationship rows are dropped before matching.

Cluster count is selected with an elbow view. The selected cluster count is 8 for this run. The elbow table is still generated for review.

The final feature audit CSV lists every feature actually used in clustering. Flattened game-matrix indicators, raw theme-name text tokens, and duplicate `product_line` are not used, so the same signal is not counted twice and raw name text does not drive clusters. Own status and game class from the accepted performance mapping are included as clustering features.

Cluster description charts show the top 2 semantic theme categories plus the top 4 other positive-lift attr features for each cluster, with user-readable labels.


In [16]:
from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import pdist, squareform

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "Data").exists() and (ROOT.parent / "Data").exists():
    ROOT = ROOT.parent

ATTR_PATH = ROOT / "Data" / "processed" / "game_attr_cleaned.csv"
ATTR_RAW_PATH = ROOT / "Data" / "raw sample" / "game_attr (2).csv"
REL_MAP_PATH = ROOT / "Data" / "processed" / "attr_clean_to_perf_best_match_mapping_all_best_candidates.csv"

OUTPUT_DIR = ROOT / "Data" / "processed" / "reports" / "attr_only_elbow_clustering"
CLUSTER_CHART_DIR = OUTPUT_DIR / "cluster_feature_charts"
HOLDOUT_CHART_DIR = OUTPUT_DIR / "holdout_feature_charts"
for path in [OUTPUT_DIR, CLUSTER_CHART_DIR, HOLDOUT_CHART_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PROFILES_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_profiles.csv"
FEATURE_MATRIX_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_features.csv"
FEATURE_USAGE_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_feature_usage.csv"
MAPPING_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_like_game_mapping.csv"
GROUPED_MAPPING_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_like_game_mapping_grouped.csv"
CLUSTER_SUMMARY_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_cluster_summary.csv"
THEME_CATEGORY_OUTPUT_PATH = ROOT / "Data" / "processed" / "attr_only_elbow_theme_categories.csv"
ELBOW_OUTPUT_PATH = OUTPUT_DIR / "elbow_scores.csv"
CHART_INDEX_OUTPUT_PATH = OUTPUT_DIR / "chart_index.csv"
MANIFEST_OUTPUT_PATH = OUTPUT_DIR / "manifest.json"

print(f"Project root: {ROOT}")
print(f"Attr input: {ATTR_PATH}")
print(f"Relationship map: {REL_MAP_PATH}")

Project root: c:\EveriTools\Workspace\Code\DemandForeCast
Attr input: c:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\game_attr_cleaned.csv
Relationship map: c:\EveriTools\Workspace\Code\DemandForeCast\Data\processed\attr_clean_to_perf_best_match_mapping_all_best_candidates.csv


In [ ]:
HOLDOUT_FRACTION = 0.10
MIN_HOLDOUT_ROWS = 1
MAX_LIKE_GAMES_PER_TARGET = None  # None keeps every historical attr game inside the smallest qualifying dendrogram cluster.

LINKAGE_METHOD = "complete"
DISTANCE_METRIC = "cosine"
ELBOW_MIN_CLUSTERS = 2
ELBOW_MAX_CLUSTERS = 16
ELBOW_SELECTED_N_CLUSTERS = 8
MAX_DENDROGRAM_LEAVES = 90
MAX_TEXT_TOKENS = 140
TOP_CLUSTER_FEATURES = 14
TOP_HOLDOUT_FEATURES = 16

EXCLUDED_MATCH_METHODS = {
    "possible_review_fuzzy",
    "weak_review_only",
    "no_match",
}

ATTR_FEATURE_MULTIPLIER = 2.0
TEXT_TOKEN_WEIGHT = 0.70
PROFILE_BIAS_WEIGHT = 0.01

In [ ]:
attr = pd.read_csv(ATTR_PATH, low_memory=False)
rel = pd.read_csv(REL_MAP_PATH, low_memory=False)

raw_extra_columns = [
    "theme_sk",
    "family_nk",
    "brand_nk",
    "brand_name",
    "cert_family_ref",
    "business_segment",
    "lines",
    "ways",
    "gaming_channel_cd",
    "volatility_cd",
    "progressive_tiers",
    "na_release_date",
    "lac_release_date",
    "emea_release_date",
    "apac_release_date",
]
if ATTR_RAW_PATH.exists():
    attr_raw = pd.read_csv(ATTR_RAW_PATH, usecols=lambda column: column in raw_extra_columns, low_memory=False)
    attr_raw = attr_raw.drop_duplicates(subset=["theme_sk"])
    extra_cols = [column for column in attr_raw.columns if column != "theme_sk"]
    attr = attr.merge(attr_raw[["theme_sk", *extra_cols]], on="theme_sk", how="left")

print(f"Attr rows: {len(attr):,}; columns after raw extras: {len(attr.columns):,}")
print(f"Relationship-map rows before filter: {len(rel):,}")
display(attr.head(3))
display(rel.head(3))

In [ ]:
def norm_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text if text else pd.NA


def clean_category(value):
    text = norm_text(value)
    if pd.isna(text):
        return "__missing__"
    text = re.sub(r"[\[\]'\"]", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "__missing__"


def numeric_series(series):
    cleaned = (
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


def bool_series(series):
    truthy = {"true", "1", "y", "yes", "t"}
    falsy = {"false", "0", "n", "no", "f"}

    def convert(value):
        if pd.isna(value):
            return np.nan
        if isinstance(value, bool):
            return 1.0 if value else 0.0
        text = str(value).strip().lower()
        if text in truthy:
            return 1.0
        if text in falsy:
            return 0.0
        number = pd.to_numeric(text, errors="coerce")
        if pd.isna(number):
            return np.nan
        return 1.0 if float(number) != 0 else 0.0

    return series.map(convert).astype(float)


def list_unique(series, limit=8):
    values = []
    seen = set()
    for value in series.dropna().astype(str):
        cleaned = value.strip()
        key = cleaned.lower()
        if not cleaned or key in seen:
            continue
        values.append(cleaned)
        seen.add(key)
        if len(values) >= limit:
            break
    return " | ".join(values)


def parse_listish(value):
    if pd.isna(value):
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except Exception:
        pass
    return [piece.strip() for piece in re.split(r"[,;|/]+", text) if piece.strip()]


def months_between(later, earlier):
    if pd.isna(later) or pd.isna(earlier):
        return np.nan
    return (later.year - earlier.year) * 12 + (later.month - earlier.month)


def release_age_bucket(months):
    if pd.isna(months):
        return "unknown_age"
    if months < 6:
        return "0_5_months"
    if months < 12:
        return "6_11_months"
    if months < 24:
        return "12_23_months"
    if months < 48:
        return "24_47_months"
    if months < 84:
        return "48_83_months"
    return "84_plus_months"


def robust_scale_01(series):
    values = numeric_series(series).astype(float)
    if values.notna().sum() == 0:
        return None
    filled = values.fillna(values.median())
    low = filled.quantile(0.01)
    high = filled.quantile(0.99)
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return None
    return ((filled.clip(low, high) - low) / (high - low)).fillna(0.0)


STOPWORDS = {
    "the", "and", "for", "with", "from", "this", "that", "game", "games", "slot", "slots",
    "deluxe", "edition", "everi", "igt", "inc", "llc", "new", "machine", "used",
    "exclude", "unknown", "missing", "theme", "standard", "video",
}


def tokenize_text(value):
    text = clean_category(value).replace("_", " ")
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return [token for token in tokens if len(token) >= 3 and token not in STOPWORDS and not token.isdigit()]


MISSING_FEATURE_VALUES = {"__missing__", "__other__", "uncategorized_theme", "missing", "nan", "none", "null", ""}
THEME_CATEGORY_COLUMN = "semantic_theme_bucket"
THEME_TEXT_COLUMNS = ["theme_name", "theme_name_friendly", "family_name", "brand_name"]


FRIENDLY_SOURCE_NAMES = {
    "source_company_cd": "Source company",
    "source_system_cd": "Source system",
    "theme_name": "Theme name",
    "theme_name_friendly": "Theme name",
    "family_name": "Family",
    "brand_name": "Brand",
    "product_family": "Product family",
    "product_segment": "Product segment",
    "product_line": "Product line",
    "parent_own_status": "Own status",
    "game_classification": "Game class",
    "game_matrix": "Game matrix",
    "business_segment": "Business segment",
    "gaming_channel_cd": "Gaming channel",
    "volatility_cd": "Volatility",
    "progressive_tiers": "Progressive tiers",
    "cert_family_ref": "Certification family",
    "attr_theme_bucket": "Theme bucket",
    "semantic_theme_bucket": "Theme category",
    "semantic_theme_keywords": "Theme keywords",
    "release_age_bucket": "Release age bucket",
    "is_wap": "WAP",
    "is_poker": "Poker",
    "is_mlp": "MLP",
    "is_multi_game": "Multi-game",
    "is_tournament_capable": "Tournament capable",
    "is_multi_denom": "Multi-denom",
    "rtp_default": "Default RTP",
    "max_bet": "Max bet",
    "min_bet": "Min bet",
    "lines": "Lines",
    "ways": "Ways",
    "release_age_months": "Release age months",
}


FRIENDLY_VALUE_LABELS = {
    "ancient_civilizations": "Ancient Civilizations",
    "fantasy_mythology": "Fantasy / Mythology",
    "nature_animals": "Nature / Animals",
    "culture_regional": "Culture / Regional",
    "entertainment_licensed": "Entertainment / Licensed",
    "adventure_exploration": "Adventure / Exploration",
    "seasonal_holiday": "Seasonal / Holiday",
    "sports_games": "Sports / Games",
    "food_drink": "Food / Drink",
    "vehicles_speed": "Vehicles / Speed",
    "mystery_scifi": "Mystery / Sci-Fi",
    "luxury_classic": "Luxury / Classic",
}


def is_missing_feature_value(value):
    return clean_category(value) in MISSING_FEATURE_VALUES


def title_preserving_acronyms(value):
    text = str(value).replace("_", " ").strip()
    if not text:
        return ""
    words = []
    acronyms = {"wap", "rtp", "mlp", "hhr", "te", "ort", "pcs", "dcx", "vlt", "vgt", "tls", "ip"}
    for word in text.split():
        stripped = word.strip()
        if stripped.lower() in acronyms:
            words.append(stripped.upper())
        else:
            words.append(stripped[:1].upper() + stripped[1:].lower())
    return " ".join(words)


def friendly_source_name(column):
    return FRIENDLY_SOURCE_NAMES.get(str(column), title_preserving_acronyms(column))


def friendly_value(value):
    text = clean_category(value)
    if text in MISSING_FEATURE_VALUES:
        return ""
    if text in FRIENDLY_VALUE_LABELS:
        return FRIENDLY_VALUE_LABELS[text]
    return title_preserving_acronyms(text)


def friendly_feature_label(source_column, feature_type, feature_value=None):
    source = friendly_source_name(source_column)
    if feature_type == "categorical":
        value = friendly_value(feature_value)
        return f"{source}: {value}" if value else source
    if feature_type == "boolean":
        return source
    if feature_type == "numeric":
        return source
    if feature_type == "text_token":
        return f"Theme text: {friendly_value(feature_value)}"
    return source


def pretty_feature_name(name):
    if "feature_usage" in globals():
        matches = feature_usage.loc[feature_usage["feature_column"].eq(name)]
        if not matches.empty:
            return str(matches.iloc[0]["friendly_feature_name"])
    text = str(name)
    for prefix in ["cat__", "bool__", "num__", "token__"]:
        text = text.replace(prefix, "")
    text = text.replace("__missing__", "missing")
    return title_preserving_acronyms(text)[:80]


def safe_filename(value, max_len=80):
    text = re.sub(r"[^a-zA-Z0-9]+", "_", str(value)).strip("_").lower()
    if not text:
        text = "item"
    return text[:max_len]


In [ ]:
rel_work = rel.copy()
rel_work["join_attr_theme_name"] = rel_work["theme_name_friendly"].map(norm_text)
rel_work["match_method_norm"] = rel_work["match_method"].map(norm_text)
rel_work["matched_historical_game_name"] = rel_work["perf_game_name"].map(norm_text)
rel_work["match_score_num"] = numeric_series(rel_work["match_score"])
rel_work["matched_release_date"] = pd.to_datetime(rel_work["game_release_date"], errors="coerce")

rel_filtered = rel_work[
    rel_work["join_attr_theme_name"].notna()
    & rel_work["matched_historical_game_name"].notna()
    & ~rel_work["match_method_norm"].isin(EXCLUDED_MATCH_METHODS)
].copy()

method_priority = {
    "raw_exact": 1,
    "basic_clean_exact": 2,
    "aggressive_clean_exact": 3,
    "very_likely_fuzzy": 4,
}
rel_filtered["method_priority"] = rel_filtered["match_method_norm"].map(method_priority).fillna(99)
rel_filtered = rel_filtered.sort_values(
    ["join_attr_theme_name", "method_priority", "match_score_num"],
    ascending=[True, True, False],
)

rel_profile = (
    rel_filtered.groupby("join_attr_theme_name", as_index=False)
    .agg(
        best_historical_game_name=("matched_historical_game_name", "first"),
        historical_game_names=("matched_historical_game_name", lambda s: list_unique(s, limit=8)),
        historical_candidate_count=("matched_historical_game_name", "nunique"),
        accepted_match_methods=("match_method_norm", lambda s: list_unique(s, limit=6)),
        best_match_method=("match_method_norm", "first"),
        best_match_score=("match_score_num", "first"),
        matched_release_date=("matched_release_date", "first"),
        parent_own_status=("parent_own_status", "first"),
        game_classification=("game_classification", "first"),
    )
)

print(f"Relationship-map rows after filter: {len(rel_filtered):,}")
print(f"Accepted attr themes in relationship map: {rel_profile['join_attr_theme_name'].nunique():,}")
print(rel_filtered["match_method_norm"].value_counts().to_string())
display(rel_profile.head(5))

In [ ]:
THEME_BUCKET_KEYWORDS = {
    "ancient_civilizations": [
        "egypt", "egyptian", "pharaoh", "pyramid", "nile", "cleopatra", "ramosis", "ramses",
        "roman", "rome", "caesar", "gladiator", "greek", "zeus", "athena", "hercules", "odyssey",
        "mayan", "aztec", "inca", "incan", "babylon", "babylonian", "emperor", "dynasty",
    ],
    "fantasy_mythology": [
        "dragon", "phoenix", "fairy", "fairies", "genie", "magic", "wizard", "witch", "myth",
        "mythology", "thor", "valkyrie", "unicorn", "mermaid", "siren", "goddess", "god", "legend",
    ],
    "nature_animals": [
        "animal", "buffalo", "wolf", "tiger", "lion", "leopard", "panther", "eagle", "bear", "horse",
        "stallion", "dolphin", "shark", "fish", "frog", "turtle", "panda", "monkey", "gorilla", "bird",
        "flamingo", "jungle", "forest", "ocean", "sea", "island", "mountain", "river", "lotus",
        "orchid", "flower", "garden", "safari",
    ],
    "culture_regional": [
        "asian", "china", "chinese", "japan", "japanese", "samurai", "ninja", "irish", "ireland",
        "italy", "italian", "latin", "mexico", "mexican", "fiesta", "western", "cowboy", "native",
        "indian", "australia", "australian", "africa", "african", "country", "mardi gras",
    ],
    "entertainment_licensed": [
        "movie", "tv", "television", "music", "show", "rock", "concert", "celebrity", "cartoon",
        "character", "beerfest", "casper", "karate kid", "willie", "nelson", "press your luck", "hollywood",
    ],
    "adventure_exploration": [
        "pirate", "viking", "treasure", "quest", "voyage", "adventure", "expedition", "frontier",
        "explorer", "journey", "trail", "wild", "outlaw",
    ],
    "seasonal_holiday": [
        "winter", "christmas", "holiday", "halloween", "spooky", "fright", "haunted", "harvest",
        "valentine", "easter", "fiesta", "firework", "fireworks",
    ],
    "sports_games": [
        "football", "baseball", "basketball", "golf", "derby", "racing", "race", "nascar", "boxing",
        "soccer", "hockey", "touchdown", "stadium", "sports",
    ],
    "food_drink": [
        "chili", "chile", "pepper", "fruit", "cherry", "lemon", "orange", "grape", "beer", "wine",
        "cocktail", "tequila", "candy", "cake", "sugar", "sweet",
    ],
    "vehicles_speed": [
        "car", "cars", "motorcycle", "bike", "truck", "train", "locomotive", "plane", "aircraft",
        "speed", "speedway", "road", "highway",
    ],
    "mystery_scifi": [
        "space", "galaxy", "cosmic", "planet", "alien", "moon", "starship", "sci fi", "scifi",
        "mystery", "horror", "zombie", "mystic", "supernova",
    ],
    "luxury_classic": [
        "gold", "golden", "diamond", "jewel", "jewels", "ruby", "emerald", "pearl", "cash", "money",
        "wealth", "wealthy", "riches", "fortune", "royal", "king", "queen", "crown", "classic", "casino",
        "jackpot", "lucky", "luck", "win", "winner", "million", "millionaire",
    ],
}


def normalized_theme_search_text(row):
    pieces = []
    for column in THEME_TEXT_COLUMNS:
        value = row.get(column, pd.NA)
        if pd.notna(value):
            text = str(value).strip().lower()
            if text and text != "nan":
                pieces.append(text)
    text = " ".join(pieces)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def theme_keyword_pattern(keyword):
    normalized = re.sub(r"[^a-z0-9]+", " ", str(keyword).lower()).strip()
    escaped = re.escape(normalized).replace(r"\ ", r"\s+")
    return rf"(?<![a-z0-9]){escaped}s?(?![a-z0-9])"


def derive_semantic_theme_bucket(row):
    text = normalized_theme_search_text(row)
    if not text:
        return "__missing__", ""
    for bucket, keywords in THEME_BUCKET_KEYWORDS.items():
        matched = [keyword for keyword in keywords if re.search(theme_keyword_pattern(keyword), text)]
        if matched:
            return bucket, ", ".join(dict.fromkeys(matched[:6]))
    return "__missing__", ""


def first_release_date(row):
    candidates = []
    for column in ["na_release_date", "lac_release_date", "emea_release_date", "apac_release_date", "matched_release_date"]:
        if column in row and pd.notna(row[column]):
            candidates.append(row[column])
    if not candidates:
        return pd.NaT
    return min(candidates)


attr_work = attr.copy()
attr_work["join_attr_theme_name"] = attr_work["theme_name_friendly"].map(norm_text)
attr_work = attr_work[attr_work["join_attr_theme_name"].notna()].copy()
if "audit_is_deleted" in attr_work.columns:
    attr_work = attr_work[~bool_series(attr_work["audit_is_deleted"]).fillna(0).astype(bool)].copy()

for column in ["na_release_date", "lac_release_date", "emea_release_date", "apac_release_date"]:
    if column in attr_work.columns:
        attr_work[column] = pd.to_datetime(attr_work[column], errors="coerce")

profiles = attr_work.merge(rel_profile, on="join_attr_theme_name", how="inner")
profiles = profiles.reset_index(drop=True)
profiles.insert(0, "profile_id", np.arange(len(profiles), dtype=int))

profiles["profile_release_date"] = profiles.apply(first_release_date, axis=1)
reference_date = profiles["profile_release_date"].max()
if pd.isna(reference_date):
    reference_date = pd.Timestamp.today().normalize()
profiles["release_age_months"] = profiles["profile_release_date"].map(lambda value: months_between(reference_date, value)).clip(lower=0)
profiles["release_age_bucket"] = profiles["release_age_months"].map(release_age_bucket)
semantic_theme = profiles.apply(
    lambda row: pd.Series(derive_semantic_theme_bucket(row), index=["semantic_theme_bucket", "semantic_theme_keywords"]),
    axis=1,
)
profiles = pd.concat([profiles, semantic_theme], axis=1)
profiles["semantic_theme_label"] = profiles["semantic_theme_bucket"].map(friendly_value)

for matrix_token in ["bingo", "hhr", "lottery", "prize", "reels"]:
    profiles[f"attr_matrix_token_{matrix_token}"] = profiles["game_matrix"].map(
        lambda value, token=matrix_token: float(any(token in item.lower() for item in parse_listish(value)))
    )

print(f"Attr-only profiles after accepted relationship join: {len(profiles):,}")
print("Semantic theme category coverage:")
print(profiles["semantic_theme_label"].replace("", "No theme category").value_counts().head(20).to_string())
display(profiles.head(5))


In [ ]:
holdout_n = max(MIN_HOLDOUT_ROWS, math.ceil(len(profiles) * HOLDOUT_FRACTION))
holdout_indices = set(
    profiles.assign(_release_sort=profiles["profile_release_date"])
    .sort_values("_release_sort", ascending=False, na_position="last")
    .head(holdout_n)
    .index
)

profiles["is_holdout"] = profiles.index.isin(holdout_indices)
profiles["is_qualified_historical"] = ~profiles["is_holdout"]
profiles["role"] = np.select(
    [profiles["is_holdout"], profiles["is_qualified_historical"]],
    ["holdout_new_game", "qualified_historical"],
    default="historical_not_qualified",
)

print(profiles["role"].value_counts().to_string())
print(
    "Holdout release range:",
    profiles.loc[profiles["is_holdout"], "profile_release_date"].min(),
    "to",
    profiles.loc[profiles["is_holdout"], "profile_release_date"].max(),
)

In [ ]:
# Clustering features are structured attr-only.
# Exclusions:
# - game_type is not used.
# - cabinet-name and performance fields are not used.
# - raw theme-name text/token features are not used.
# - flattened game_matrix indicators are not used with raw game_matrix, so the same signal is not counted twice.
# - one clean semantic theme category is included from true theme/name fields only.
ATTR_CATEGORICAL_WEIGHTS = {
    "source_company_cd": 0.7,
    "source_system_cd": 0.35,
    "family_name": 1.7,
    "brand_name": 1.5,
    "semantic_theme_bucket": 1.4,
    "product_family": 1.0,
    "parent_own_status": 1.3,
    "game_classification": 1.5,
    "game_matrix": 2.5,
    "product_segment": 1.8,
    "business_segment": 1.4,
    "gaming_channel_cd": 1.0,
    "volatility_cd": 1.2,
    "progressive_tiers": 1.1,
    "cert_family_ref": 0.9,
    "release_age_bucket": 0.30,
}

ATTR_BOOLEAN_WEIGHTS = {
    "is_wap": 2.2,
    "is_poker": 1.9,
    "is_mlp": 1.6,
    "is_multi_game": 2.6,
    "is_tournament_capable": 1.2,
    "is_multi_denom": 1.9,
}

ATTR_NUMERIC_WEIGHTS = {
    "rtp_default": 0.9,
    "max_bet": 1.2,
    "min_bet": 0.9,
    "lines": 1.0,
    "ways": 1.0,
    "release_age_months": 0.30,
}


def add_categorical_block(frame, column, weight, max_levels=180, min_count=2, drop_missing_features=False):
    if column not in frame.columns:
        return None, []
    series = frame[column].map(clean_category).astype("string")
    counts = series.value_counts(dropna=False)
    if len(counts) <= 1:
        return None, []
    keep = set(counts[counts >= min_count].head(max_levels).index)
    bucketed = series.where(series.isin(keep), "__other__")
    dummies = pd.get_dummies(bucketed, prefix=f"cat__{column}", dtype=float)
    prefix = f"cat__{column}_"
    feature_values = {
        feature: feature[len(prefix):] if feature.startswith(prefix) else feature
        for feature in dummies.columns
    }
    if drop_missing_features:
        keep_features = [
            feature for feature, value in feature_values.items()
            if not is_missing_feature_value(value)
        ]
        dummies = dummies[keep_features]
    if dummies.shape[1] == 0:
        return None, []
    weighted = dummies * float(weight) * ATTR_FEATURE_MULTIPLIER
    rows = []
    for feature in weighted.columns:
        rows.append({
            "feature_column": feature,
            "source_column": column,
            "feature_type": "categorical",
            "feature_value": feature_values[feature],
            "base_weight": float(weight),
            "multiplier": ATTR_FEATURE_MULTIPLIER,
            "final_weight": float(weight) * ATTR_FEATURE_MULTIPLIER,
        })
    return weighted, rows


def add_boolean_block(frame, column, weight):
    if column not in frame.columns:
        return None, []
    values = bool_series(frame[column]).fillna(0.0)
    if values.nunique(dropna=False) <= 1:
        return None, []
    feature = f"bool__{column}"
    block = pd.DataFrame({feature: values * float(weight) * ATTR_FEATURE_MULTIPLIER}, index=frame.index)
    rows = [{
        "feature_column": feature,
        "source_column": column,
        "feature_type": "boolean",
        "feature_value": "true",
        "base_weight": float(weight),
        "multiplier": ATTR_FEATURE_MULTIPLIER,
        "final_weight": float(weight) * ATTR_FEATURE_MULTIPLIER,
    }]
    return block, rows


def add_numeric_block(frame, column, weight):
    if column not in frame.columns:
        return None, []
    values = numeric_series(frame[column])
    if any(token in column for token in ["bet", "lines", "ways", "months"]):
        values = np.log1p(values.clip(lower=0))
    scaled = robust_scale_01(values)
    if scaled is None or scaled.nunique(dropna=False) <= 1:
        return None, []
    feature = f"num__{column}"
    block = pd.DataFrame({feature: scaled * float(weight) * ATTR_FEATURE_MULTIPLIER}, index=frame.index)
    rows = [{
        "feature_column": feature,
        "source_column": column,
        "feature_type": "numeric",
        "feature_value": "",
        "base_weight": float(weight),
        "multiplier": ATTR_FEATURE_MULTIPLIER,
        "final_weight": float(weight) * ATTR_FEATURE_MULTIPLIER,
    }]
    return block, rows


def build_weighted_feature_matrix(frame):
    blocks = []
    feature_sources = {}
    metadata_rows = []

    for column, weight in ATTR_CATEGORICAL_WEIGHTS.items():
        block, rows = add_categorical_block(
            frame,
            column,
            weight,
            drop_missing_features=column == THEME_CATEGORY_COLUMN,
        )
        if block is not None:
            blocks.append(block)
            metadata_rows.extend(rows)
            source_family = "semantic_theme_attr" if column == THEME_CATEGORY_COLUMN else "attr"
            feature_sources.update({feature: source_family for feature in block.columns})

    for column, weight in ATTR_BOOLEAN_WEIGHTS.items():
        block, rows = add_boolean_block(frame, column, weight)
        if block is not None:
            blocks.append(block)
            metadata_rows.extend(rows)
            feature_sources.update({feature: "attr" for feature in block.columns})

    for column, weight in ATTR_NUMERIC_WEIGHTS.items():
        block, rows = add_numeric_block(frame, column, weight)
        if block is not None:
            blocks.append(block)
            metadata_rows.extend(rows)
            feature_sources.update({feature: "attr" for feature in block.columns})

    if not blocks:
        raise ValueError("No usable structured attr clustering features were created.")

    matrix = pd.concat(blocks, axis=1).fillna(0.0)
    matrix["__profile_bias"] = PROFILE_BIAS_WEIGHT
    feature_sources["__profile_bias"] = "bias"
    metadata_rows.append({
        "feature_column": "__profile_bias",
        "source_column": "profile_bias",
        "feature_type": "bias",
        "feature_value": "",
        "base_weight": PROFILE_BIAS_WEIGHT,
        "multiplier": 1.0,
        "final_weight": PROFILE_BIAS_WEIGHT,
    })
    metadata = pd.DataFrame(metadata_rows)
    return matrix, feature_sources, metadata


feature_matrix, feature_sources, feature_usage = build_weighted_feature_matrix(profiles)
feature_usage["friendly_source_name"] = feature_usage["source_column"].map(friendly_source_name)
feature_usage["friendly_feature_name"] = feature_usage.apply(
    lambda row: friendly_feature_label(row["source_column"], row["feature_type"], row["feature_value"]),
    axis=1,
)
feature_usage["feature_group"] = np.where(
    feature_usage["source_column"].eq(THEME_CATEGORY_COLUMN),
    "semantic_theme_category",
    feature_usage["feature_type"],
)
feature_usage["is_missing_or_other"] = feature_usage["feature_value"].map(is_missing_feature_value)
feature_usage["is_reportable"] = ~feature_usage["is_missing_or_other"] & feature_usage["feature_type"].ne("bias")
feature_usage["active_count"] = [
    int((feature_matrix[column] > 0).sum()) for column in feature_usage["feature_column"]
]
feature_usage["active_rate"] = feature_usage["active_count"] / len(feature_matrix)
feature_usage["mean_weighted_value"] = [
    float(feature_matrix[column].mean()) for column in feature_usage["feature_column"]
]
feature_usage["max_weighted_value"] = [
    float(feature_matrix[column].max()) for column in feature_usage["feature_column"]
]
feature_usage = feature_usage[
    [
        "feature_column",
        "friendly_feature_name",
        "source_column",
        "friendly_source_name",
        "feature_group",
        "feature_type",
        "feature_value",
        "base_weight",
        "multiplier",
        "final_weight",
        "active_count",
        "active_rate",
        "mean_weighted_value",
        "max_weighted_value",
        "is_missing_or_other",
        "is_reportable",
    ]
].sort_values(["feature_group", "feature_type", "source_column", "feature_value"]).reset_index(drop=True)

for forbidden in ["game_type", "cabinet", "performance", "perf_", "slot_cabinet", "theme_name_text", "token__", "attr_theme_bucket", "format_non_theme", "product_line"]:
    bad = [column for column in feature_matrix.columns if forbidden in column.lower()]
    if bad:
        raise ValueError(f"Forbidden clustering features found for {forbidden}: {bad[:10]}")

print(f"Feature matrix shape: {feature_matrix.shape[0]:,} profiles x {feature_matrix.shape[1]:,} structured attr-only weighted features")
print(pd.Series(feature_sources).value_counts().to_string())
print(f"Zero-norm rows: {(np.linalg.norm(feature_matrix.to_numpy(dtype=float), axis=1) == 0).sum()}")
print(f"Reportable clustering features: {int(feature_usage['is_reportable'].sum()):,}")
display(feature_usage.head(25))
display(feature_matrix.head(3))


In [ ]:
X = feature_matrix.to_numpy(dtype=float)
condensed_distances = pdist(X, metric=DISTANCE_METRIC)
if not np.isfinite(condensed_distances).all():
    raise ValueError("Pairwise distances contain NaN or infinite values.")

distance_matrix = squareform(condensed_distances)
linkage_matrix = linkage(condensed_distances, method=LINKAGE_METHOD, optimal_ordering=True)

row_norms = np.linalg.norm(X, axis=1, keepdims=True)
X_unit = X / np.where(row_norms == 0, 1, row_norms)


def within_cluster_sse(x_values, labels):
    total = 0.0
    for label in np.unique(labels):
        cluster = x_values[labels == label]
        if len(cluster) == 0:
            continue
        center = cluster.mean(axis=0, keepdims=True)
        total += float(((cluster - center) ** 2).sum())
    return total


def elbow_line_distance(xs, ys):
    points = np.column_stack([xs, ys])
    start = points[0]
    end = points[-1]
    line = end - start
    denom = np.linalg.norm(line)
    if denom == 0:
        return np.zeros(len(points))
    return np.abs(np.cross(line, start - points)) / denom


rows = []
max_k = min(ELBOW_MAX_CLUSTERS, len(profiles) - 1)
for requested_k in range(ELBOW_MIN_CLUSTERS, max_k + 1):
    labels = fcluster(linkage_matrix, t=requested_k, criterion="maxclust")
    actual_k = len(np.unique(labels))
    sse = within_cluster_sse(X_unit, labels)
    counts = pd.Series(labels).value_counts()
    rows.append({
        "requested_k": requested_k,
        "actual_k": actual_k,
        "within_cluster_sse": sse,
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
    })

elbow_scores = pd.DataFrame(rows)
elbow_scores["sse_drop"] = elbow_scores["within_cluster_sse"].shift(1) - elbow_scores["within_cluster_sse"]
distances_to_line = elbow_line_distance(
    elbow_scores["actual_k"].to_numpy(dtype=float),
    elbow_scores["within_cluster_sse"].to_numpy(dtype=float),
)
elbow_scores["elbow_distance_to_line"] = distances_to_line
auto_elbow_k = int(elbow_scores.loc[elbow_scores["elbow_distance_to_line"].idxmax(), "actual_k"])

selected_k = ELBOW_SELECTED_N_CLUSTERS
if selected_k < ELBOW_MIN_CLUSTERS or selected_k > max_k:
    raise ValueError(f"ELBOW_SELECTED_N_CLUSTERS={selected_k} is outside the valid range.")

labels = fcluster(linkage_matrix, t=selected_k, criterion="maxclust")
counts = pd.Series(labels).value_counts().sort_values(ascending=False)
remap = {old_label: new_label for new_label, old_label in enumerate(counts.index, start=1)}
profiles["cluster_id"] = np.array([remap[label] for label in labels], dtype=int)

print(f"Auto elbow suggestion: {auto_elbow_k}")
print(f"Selected clusters used for this run: {profiles['cluster_id'].nunique()} (ELBOW_SELECTED_N_CLUSTERS={ELBOW_SELECTED_N_CLUSTERS})")
display(elbow_scores)
display(profiles["cluster_id"].value_counts().sort_index().rename("profiles_per_cluster"))

In [ ]:
def find_holdout_like_games(profiles, linkage_matrix, distance_matrix):
    n = len(profiles)
    target_indices = profiles.index[profiles["is_holdout"]].tolist()
    historical_indices = set(profiles.index[profiles["is_qualified_historical"]].tolist())

    clusters = {idx: {idx} for idx in range(n)}
    merge_records = []
    for merge_step, row in enumerate(linkage_matrix, start=1):
        left = int(row[0])
        right = int(row[1])
        members = clusters[left] | clusters[right]
        node_id = n + merge_step - 1
        clusters[node_id] = members
        merge_records.append({
            "merge_step": merge_step,
            "node_id": node_id,
            "merge_distance": float(row[2]),
            "members": members,
        })

    rows = []
    for target_idx in target_indices:
        selected = None
        for record in merge_records:
            if target_idx not in record["members"]:
                continue
            qualifying = sorted(record["members"] & historical_indices)
            if qualifying:
                selected = {**record, "qualifying": qualifying}
                break

        if selected is None:
            continue

        ranked = sorted(selected["qualifying"], key=lambda candidate_idx: distance_matrix[target_idx, candidate_idx])
        if MAX_LIKE_GAMES_PER_TARGET is not None:
            ranked = ranked[:MAX_LIKE_GAMES_PER_TARGET]

        target = profiles.loc[target_idx]
        for rank, like_idx in enumerate(ranked, start=1):
            like = profiles.loc[like_idx]
            pair_distance = float(distance_matrix[target_idx, like_idx])
            similarity_score = float(np.clip(1.0 - pair_distance, 0.0, 1.0))
            merge_similarity = float(np.clip(1.0 - selected["merge_distance"], 0.0, 1.0))
            rows.append({
                "target_profile_id": int(target["profile_id"]),
                "target_theme_sk": target.get("theme_sk"),
                "target_theme_name_friendly": target.get("theme_name_friendly"),
                "target_semantic_theme_bucket": target.get("semantic_theme_bucket"),
                "target_semantic_theme_label": target.get("semantic_theme_label"),
                "target_product_family": target.get("product_family"),
                "target_product_line": target.get("product_line"),
                "target_own_status": target.get("parent_own_status"),
                "target_game_class": target.get("game_classification"),
                "target_game_type": target.get("game_type"),
                "target_game_matrix": target.get("game_matrix"),
                "target_release_date": target.get("profile_release_date"),
                "target_cluster_id": int(target.get("cluster_id")),
                "target_historical_game_name": target.get("best_historical_game_name"),
                "like_profile_id": int(like["profile_id"]),
                "like_theme_sk": like.get("theme_sk"),
                "like_theme_name_friendly": like.get("theme_name_friendly"),
                "like_semantic_theme_bucket": like.get("semantic_theme_bucket"),
                "like_semantic_theme_label": like.get("semantic_theme_label"),
                "like_product_family": like.get("product_family"),
                "like_product_line": like.get("product_line"),
                "like_own_status": like.get("parent_own_status"),
                "like_game_class": like.get("game_classification"),
                "like_game_type": like.get("game_type"),
                "like_game_matrix": like.get("game_matrix"),
                "like_release_date": like.get("profile_release_date"),
                "like_cluster_id": int(like.get("cluster_id")),
                "like_historical_game_name": like.get("best_historical_game_name"),
                "candidate_rank": rank,
                "pair_cosine_distance": pair_distance,
                "similarity_score": similarity_score,
                "merge_distance": selected["merge_distance"],
                "merge_similarity_score": merge_similarity,
                "smallest_qualifying_cluster_node_id": int(selected["node_id"]),
                "smallest_qualifying_cluster_size": len(selected["members"]),
                "qualifying_historical_in_cluster": len(selected["qualifying"]),
                "same_flat_cluster": bool(target.get("cluster_id") == like.get("cluster_id")),
            })

    return pd.DataFrame(rows)


like_games = find_holdout_like_games(profiles, linkage_matrix, distance_matrix)
if like_games.empty:
    raise ValueError("No like-game mappings were produced.")

grouped_like_games = (
    like_games.sort_values(["target_profile_id", "candidate_rank"])
    .groupby([
        "target_profile_id",
        "target_theme_sk",
        "target_theme_name_friendly",
        "target_semantic_theme_bucket",
        "target_semantic_theme_label",
        "target_product_family",
        "target_product_line",
        "target_own_status",
        "target_game_class",
        "target_game_type",
        "target_release_date",
        "target_cluster_id",
    ], dropna=False)
    .agg(
        like_game_count=("like_profile_id", "size"),
        best_similarity_score=("similarity_score", "max"),
        mean_similarity_score=("similarity_score", "mean"),
        like_attr_themes=("like_theme_name_friendly", lambda values: " | ".join(values.astype(str))),
        like_theme_categories=("like_semantic_theme_label", lambda values: " | ".join(values.fillna("").astype(str))),
        like_historical_games=("like_historical_game_name", lambda values: " | ".join(values.astype(str))),
        like_profile_ids=("like_profile_id", lambda values: " | ".join(values.astype(str))),
    )
    .reset_index()
    .sort_values(["best_similarity_score", "like_game_count"], ascending=[False, False])
)

print(f"Mapping rows: {len(like_games):,}")
print(f"Mapped holdout targets: {like_games['target_profile_id'].nunique():,} / {profiles['is_holdout'].sum():,}")
display(like_games.sort_values(["target_profile_id", "candidate_rank"]).head(30))
display(grouped_like_games.head(20))

In [ ]:
def top_values(series, n=3):
    cleaned = series.map(clean_category)
    cleaned = cleaned[~cleaned.isin(MISSING_FEATURE_VALUES)]
    counts = cleaned.value_counts(normalize=True).head(n)
    if counts.empty:
        return ""
    return "; ".join(f"{friendly_value(idx)} ({share:.0%})" for idx, share in counts.items())


def reportable_lift_from(feature_lift_frame, cluster_id, n, source_column=None, exclude_source_column=None, important_only=True):
    usage = feature_usage[feature_usage["is_reportable"]].copy()
    if source_column is not None:
        usage = usage[usage["source_column"].eq(source_column)]
    if exclude_source_column is not None:
        usage = usage[~usage["source_column"].eq(exclude_source_column)]
    reportable = usage["feature_column"].tolist()
    if not reportable:
        return pd.Series(dtype=float)
    lifted_all = feature_lift_frame.loc[cluster_id, reportable].sort_values(ascending=False)
    lifted_all = lifted_all[lifted_all > 0]
    if lifted_all.empty:
        return lifted_all

    if not important_only:
        return lifted_all.head(n)

    top_lift = float(lifted_all.iloc[0])
    threshold = max(
        CLUSTER_FEATURE_MIN_ABSOLUTE_LIFT,
        top_lift * CLUSTER_FEATURE_MIN_RELATIVE_TO_TOP,
    )
    important = lifted_all[lifted_all >= threshold]
    if len(important) >= CLUSTER_FEATURE_MIN_COUNT:
        return important.head(n)

    filler = lifted_all.drop(index=important.index, errors="ignore")
    combined = pd.concat([important, filler]).head(min(n, CLUSTER_FEATURE_MIN_COUNT))
    return combined


def cluster_description_lifts(cluster_id, theme_count=2, attr_count=4, feature_lift_frame=None):
    if feature_lift_frame is None:
        feature_lift_frame = feature_lift
    theme_lift = reportable_lift_from(
        feature_lift_frame,
        cluster_id,
        theme_count,
        source_column=THEME_CATEGORY_COLUMN,
        important_only=False,
    )
    attr_lift = reportable_lift_from(
        feature_lift_frame,
        cluster_id,
        attr_count,
        exclude_source_column=THEME_CATEGORY_COLUMN,
        important_only=False,
    )
    combined = pd.concat([theme_lift, attr_lift])
    combined = combined[~combined.index.duplicated(keep="first")]
    return theme_lift, attr_lift, combined


def build_cluster_summary(profiles, feature_matrix):
    feature_view = feature_matrix.drop(columns=["__profile_bias"], errors="ignore")
    cluster_means = feature_view.groupby(profiles["cluster_id"]).mean()
    global_means = feature_view.mean()
    feature_lift = cluster_means.subtract(global_means, axis=1)

    summary = (
        profiles.groupby("cluster_id", as_index=False)
        .agg(
            profile_count=("profile_id", "size"),
            holdout_count=("is_holdout", "sum"),
            historical_count=("is_qualified_historical", "sum"),
            min_release_date=("profile_release_date", "min"),
            max_release_date=("profile_release_date", "max"),
            median_release_age_months=("release_age_months", "median"),
        )
        .sort_values("cluster_id")
    )

    characteristic_columns = [
        "semantic_theme_bucket",
        "product_family",
        "parent_own_status",
        "game_classification",
        "game_matrix",
        "product_segment",
        "family_name",
        "brand_name",
        "business_segment",
        "volatility_cd",
        "is_multi_denom",
        "is_wap",
        "is_poker",
    ]
    for column in characteristic_columns:
        if column in profiles.columns:
            values = profiles.groupby("cluster_id")[column].apply(top_values).rename(f"top_{column}")
            summary = summary.merge(values.reset_index(), on="cluster_id", how="left")

    top_theme_features = []
    top_attr_features = []
    top_features = []
    for cluster_id in summary["cluster_id"]:
        theme_lift, attr_lift, combined_lift = cluster_description_lifts(cluster_id, feature_lift_frame=feature_lift)
        theme_text = "; ".join(pretty_feature_name(name) for name in theme_lift.index[:2])
        attr_text = "; ".join(pretty_feature_name(name) for name in attr_lift.index[:4])
        top_theme_features.append(theme_text)
        top_attr_features.append(attr_text)
        parts = []
        if theme_text:
            parts.append(f"Theme categories: {theme_text}")
        if attr_text:
            parts.append(f"Attr features: {attr_text}")
        if not parts:
            parts.append("No positive-lift reportable features")
        top_features.append(" | ".join(parts))
    summary["top_theme_category_features"] = top_theme_features
    summary["top_structured_attr_features"] = top_attr_features
    summary["top_distinguishing_features"] = top_features
    return summary, feature_lift


cluster_summary, feature_lift = build_cluster_summary(profiles, feature_matrix)
display(cluster_summary)


In [ ]:
def plot_elbow(elbow_scores):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(elbow_scores["actual_k"], elbow_scores["within_cluster_sse"], marker="o", linewidth=2)
    auto_k = int(elbow_scores.loc[elbow_scores["elbow_distance_to_line"].idxmax(), "actual_k"])
    selected_k = profiles["cluster_id"].nunique()
    ax.axvline(auto_k, color="#777777", linestyle="--", linewidth=1.2, label=f"Auto elbow: {auto_k}")
    ax.axvline(selected_k, color="#111111", linestyle="-", linewidth=1.4, label=f"Selected: {selected_k}")
    ax.set_title("Elbow Method On Attr-Only Features")
    ax.set_xlabel("Number of clusters")
    ax.set_ylabel("Within-cluster SSE on normalized attr matrix")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    path = OUTPUT_DIR / "elbow_method.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return path


def classical_mds(distance_matrix, n_components=2):
    n = distance_matrix.shape[0]
    squared = distance_matrix ** 2
    centering = np.eye(n) - np.ones((n, n)) / n
    gram = -0.5 * centering @ squared @ centering
    eigvals, eigvecs = np.linalg.eigh(gram)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    positive = np.maximum(eigvals[:n_components], 0)
    coords = eigvecs[:, :n_components] * np.sqrt(positive)
    if coords.shape[1] < n_components:
        coords = np.pad(coords, ((0, 0), (0, n_components - coords.shape[1])))
    return coords


def plot_dendrogram_view():
    selected_k = profiles["cluster_id"].nunique()
    cut_distance = linkage_matrix[-(selected_k - 1), 2] if selected_k > 1 else linkage_matrix[-1, 2]
    labels = [
        f"{int(row.profile_id)} | {str(row.theme_name_friendly)[:34]} | {str(row.product_family)[:20]}"
        for row in profiles.itertuples(index=False)
    ]
    fig, ax = plt.subplots(figsize=(18, 9))
    kwargs = {
        "Z": linkage_matrix,
        "labels": labels,
        "leaf_rotation": 90,
        "leaf_font_size": 7,
        "color_threshold": cut_distance,
        "show_contracted": True,
        "ax": ax,
    }
    if len(profiles) > MAX_DENDROGRAM_LEAVES:
        kwargs.update({"truncate_mode": "lastp", "p": MAX_DENDROGRAM_LEAVES})
    dendrogram(**kwargs)
    ax.axhline(cut_distance, color="black", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_title("Attr-Only Complete-Linkage Dendrogram")
    ax.set_xlabel("Attr theme profile")
    ax.set_ylabel(f"{DISTANCE_METRIC.title()} distance")
    path = OUTPUT_DIR / "dendrogram_selected_clusters.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return path


def plot_projection():
    coords = classical_mds(distance_matrix)
    profiles["mds_x"] = coords[:, 0]
    profiles["mds_y"] = coords[:, 1]
    fig, ax = plt.subplots(figsize=(10, 7))
    cmap = plt.get_cmap("tab10")
    for idx, cluster_id in enumerate(sorted(profiles["cluster_id"].unique())):
        mask = profiles["cluster_id"].eq(cluster_id)
        historical = mask & ~profiles["is_holdout"]
        holdout = mask & profiles["is_holdout"]
        color = cmap(idx % cmap.N)
        ax.scatter(profiles.loc[historical, "mds_x"], profiles.loc[historical, "mds_y"], s=24, alpha=0.50, color=color, label=f"Cluster {cluster_id}")
        ax.scatter(profiles.loc[holdout, "mds_x"], profiles.loc[holdout, "mds_y"], s=72, marker="x", linewidths=1.7, color=color)
    ax.set_title("Attr-Only 2D Projection (x = Holdout)")
    ax.set_xlabel("MDS 1")
    ax.set_ylabel("MDS 2")
    ax.grid(alpha=0.20)
    ax.legend(ncol=2, fontsize=8, frameon=False)
    path = OUTPUT_DIR / "cluster_projection_mds.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return path


elbow_plot_path = plot_elbow(elbow_scores)
dendrogram_path = plot_dendrogram_view()
projection_path = plot_projection()
print("Saved overview visuals:")
for path in [elbow_plot_path, dendrogram_path, projection_path]:
    print(f"- {path}")

In [ ]:
def save_cluster_feature_chart(cluster_id):
    theme_lift, attr_lift, lifted = cluster_description_lifts(cluster_id)
    labels = [pretty_feature_name(name) for name in lifted.index]
    values = lifted.to_numpy()
    sources = feature_usage.set_index("feature_column").reindex(lifted.index)["source_column"] if len(lifted) else pd.Series(dtype="object")
    colors = ["#7b5ea7" if source == THEME_CATEGORY_COLUMN else "#3568a8" for source in sources]
    cluster_rows = profiles[profiles["cluster_id"].eq(cluster_id)]
    summary_row = cluster_summary[cluster_summary["cluster_id"].eq(cluster_id)].iloc[0]

    fig, ax = plt.subplots(figsize=(12, 6.5))
    if len(values):
        y = np.arange(len(values))
        ax.barh(y, values, color=colors)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=9)
        ax.invert_yaxis()
    else:
        ax.text(0.5, 0.5, "No positive-lift reportable features", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("Feature lift vs global mean")
    ax.text(
        0.99,
        0.02,
        "Purple = top theme categories; blue = top other attr features",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
        color="#555555",
    )
    ax.set_title(
        f"Cluster {cluster_id}: Top 2 Theme Categories + Top 4 Attr Features "
        f"({len(cluster_rows)} games, {int(summary_row['holdout_count'])} holdout)"
    )
    ax.grid(axis="x", alpha=0.20)

    text_lines = [
        f"Top semantic themes: {summary_row.get('top_semantic_theme_bucket', '')}",
        f"Top product family: {summary_row.get('top_product_family', '')}",
        f"Top own status: {summary_row.get('top_parent_own_status', '')}",
        f"Top game class: {summary_row.get('top_game_classification', '')}",
        f"Top game matrix: {summary_row.get('top_game_matrix', '')}",
    ]
    fig.text(0.02, 0.02, "\n".join([line for line in text_lines if not line.endswith(': ')]), ha="left", va="bottom", fontsize=9)
    path = CLUSTER_CHART_DIR / f"cluster_{int(cluster_id):02d}_features.png"
    fig.subplots_adjust(left=0.37, right=0.98, top=0.88, bottom=0.20)
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


for old_chart in CLUSTER_CHART_DIR.glob("*.png"):
    old_chart.unlink()

cluster_chart_paths = []
for cluster_id in sorted(profiles["cluster_id"].unique()):
    cluster_chart_paths.append(save_cluster_feature_chart(cluster_id))

print(f"Saved {len(cluster_chart_paths)} cluster feature charts to {CLUSTER_CHART_DIR}")
display(pd.DataFrame({"cluster_id": sorted(profiles["cluster_id"].unique()), "chart_path": [str(p) for p in cluster_chart_paths]}))


In [ ]:
def top_active_features(profile_index, n=TOP_HOLDOUT_FEATURES):
    values = feature_matrix.iloc[profile_index].drop(labels=["__profile_bias"], errors="ignore")
    active = values[values > 0].sort_values(ascending=False).head(n)
    return active


def save_holdout_feature_chart(profile_index):
    row = profiles.loc[profile_index]
    active = top_active_features(profile_index)
    target_rows = like_games[like_games["target_profile_id"].eq(row["profile_id"])].sort_values("candidate_rank").head(5)
    like_text = "; ".join(
        f"{item.like_theme_name_friendly} ({item.similarity_score:.3f})"
        for item in target_rows.itertuples(index=False)
    )
    if not like_text:
        like_text = "No like games found"

    fig, (ax_text, ax_bar) = plt.subplots(
        2,
        1,
        figsize=(12, 8),
        gridspec_kw={"height_ratios": [1.1, 2.2]},
    )
    ax_text.axis("off")
    text = (
        f"Holdout: {row.get('theme_name_friendly')}\\n"
        f"Assigned cluster: {int(row.get('cluster_id'))}\\n"
        f"Product family: {row.get('product_family')} | Product line: {row.get('product_line')}\\n"
        f"Game type: {row.get('game_type')} | Game matrix: {row.get('game_matrix')}\\n"
        f"Release date: {row.get('profile_release_date')} | Historical mapping: {row.get('best_historical_game_name')}\\n"
        f"Top like games: {like_text}"
    )
    ax_text.text(0.01, 0.98, text, va="top", ha="left", fontsize=10, wrap=True)

    labels = [pretty_feature_name(name) for name in active.index]
    values = active.to_numpy()
    y = np.arange(len(values))
    ax_bar.barh(y, values, color="#2f7f63")
    ax_bar.set_yticks(y)
    ax_bar.set_yticklabels(labels, fontsize=9)
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel("Weighted active attr feature value")
    ax_bar.set_title("Strongest Active Attr Features For This Holdout")
    ax_bar.grid(axis="x", alpha=0.20)

    file_name = f"holdout_{int(row['profile_id']):04d}_cluster_{int(row['cluster_id']):02d}_{safe_filename(row.get('theme_name_friendly'))}.png"
    path = HOLDOUT_CHART_DIR / file_name
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


holdout_chart_rows = []
for profile_index in profiles.index[profiles["is_holdout"]]:
    path = save_holdout_feature_chart(profile_index)
    row = profiles.loc[profile_index]
    holdout_chart_rows.append({
        "profile_id": int(row["profile_id"]),
        "theme_name_friendly": row.get("theme_name_friendly"),
        "semantic_theme_label": row.get("semantic_theme_label"),
        "cluster_id": int(row.get("cluster_id")),
        "chart_path": str(path),
    })

holdout_chart_index = pd.DataFrame(holdout_chart_rows).sort_values(["cluster_id", "theme_name_friendly"])
print(f"Saved {len(holdout_chart_index)} holdout feature charts to {HOLDOUT_CHART_DIR}")
display(holdout_chart_index.head(20))

In [ ]:
theme_category_columns = [
    "profile_id",
    "theme_sk",
    "theme_name_friendly",
    "family_name",
    "brand_name",
    "semantic_theme_bucket",
    "semantic_theme_label",
    "semantic_theme_keywords",
    "cluster_id",
    "is_holdout",
    "is_qualified_historical",
]
theme_category_audit = profiles[[column for column in theme_category_columns if column in profiles.columns]].copy()

theme_category_audit.to_csv(THEME_CATEGORY_OUTPUT_PATH, index=False)
profiles.to_csv(PROFILES_OUTPUT_PATH, index=False)
feature_matrix.to_csv(FEATURE_MATRIX_OUTPUT_PATH, index=False)
feature_usage.to_csv(FEATURE_USAGE_OUTPUT_PATH, index=False)
like_games.to_csv(MAPPING_OUTPUT_PATH, index=False)
grouped_like_games.to_csv(GROUPED_MAPPING_OUTPUT_PATH, index=False)
cluster_summary.to_csv(CLUSTER_SUMMARY_OUTPUT_PATH, index=False)
elbow_scores.to_csv(ELBOW_OUTPUT_PATH, index=False)
holdout_chart_index.to_csv(CHART_INDEX_OUTPUT_PATH, index=False)

manifest = {
    "inputs": {
        "attr_path": str(ATTR_PATH),
        "attr_raw_path": str(ATTR_RAW_PATH),
        "rel_map_path": str(REL_MAP_PATH),
    },
    "profile_rows": int(len(profiles)),
    "feature_columns": int(feature_matrix.shape[1]),
    "feature_source_counts": {str(k): int(v) for k, v in pd.Series(feature_sources).value_counts().items()},
    "reportable_feature_columns": int(feature_usage["is_reportable"].sum()),
    "holdout_profiles": int(profiles["is_holdout"].sum()),
    "historical_profiles": int(profiles["is_qualified_historical"].sum()),
    "mapped_holdout_profiles": int(like_games["target_profile_id"].nunique()),
    "mapping_rows": int(len(like_games)),
    "auto_elbow_k": int(auto_elbow_k),
    "selected_clusters": int(profiles["cluster_id"].nunique()),
    "selected_clusters_setting": int(ELBOW_SELECTED_N_CLUSTERS),
    "excluded_match_methods": sorted(EXCLUDED_MATCH_METHODS),
    "distance_metric": DISTANCE_METRIC,
    "linkage_method": LINKAGE_METHOD,
    "included_derived_features": [
        "semantic_theme_bucket_from_theme_name_family_brand_only",
    ],
    "excluded_feature_families": [
        "cabinet_name",
        "performance",
        "game_type",
        "flattened_game_matrix_duplicates",
        "math_model_family_duplicate",
        "product_line_duplicate_of_product_family",
        "raw_theme_text_tokens",
        "attr_theme_bucket_format_non_theme",
    ],
    "outputs": {
        "profiles": str(PROFILES_OUTPUT_PATH),
        "theme_categories": str(THEME_CATEGORY_OUTPUT_PATH),
        "feature_matrix": str(FEATURE_MATRIX_OUTPUT_PATH),
        "feature_usage": str(FEATURE_USAGE_OUTPUT_PATH),
        "mapping": str(MAPPING_OUTPUT_PATH),
        "grouped_mapping": str(GROUPED_MAPPING_OUTPUT_PATH),
        "cluster_summary": str(CLUSTER_SUMMARY_OUTPUT_PATH),
        "elbow_scores": str(ELBOW_OUTPUT_PATH),
        "chart_index": str(CHART_INDEX_OUTPUT_PATH),
        "cluster_chart_dir": str(CLUSTER_CHART_DIR),
        "holdout_chart_dir": str(HOLDOUT_CHART_DIR),
        "report_dir": str(OUTPUT_DIR),
    },
}
MANIFEST_OUTPUT_PATH.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

print("Saved outputs:")
for path in [
    THEME_CATEGORY_OUTPUT_PATH,
    PROFILES_OUTPUT_PATH,
    FEATURE_MATRIX_OUTPUT_PATH,
    FEATURE_USAGE_OUTPUT_PATH,
    MAPPING_OUTPUT_PATH,
    GROUPED_MAPPING_OUTPUT_PATH,
    CLUSTER_SUMMARY_OUTPUT_PATH,
    ELBOW_OUTPUT_PATH,
    CHART_INDEX_OUTPUT_PATH,
    MANIFEST_OUTPUT_PATH,
]:
    print(f"- {path}")
